# Read the Mongo catalog sync table

This notebook reads the Delta table created by the MongoDB-to-catalog sync job.

The YAML config uses `catalog_table: lakehouse.crawler.responses_raw`

airflow/dags/mongo_catalog_sync_jobs.yaml


In [1]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = (
    SparkSession.builder.appName('mongo-catalog-read')
    .master('spark://spark-master:7077')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.hive.metastore.uris', 'thrift://hive-metastore:9083')
    .config('spark.sql.warehouse.dir', 's3a://warehouse/spark-warehouse')
    .config('spark.jars.ivy', '/tmp/.ivy2')
    .config('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000')
    .config('spark.hadoop.fs.s3a.access.key', 'minioadmin')
    .config('spark.hadoop.fs.s3a.secret.key', 'minioadmin')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.hadoop.fs.s3a.endpoint.region', 'us-east-1')
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
)

spark = configure_spark_with_delta_pip(builder).enableHiveSupport().getOrCreate()


In [9]:
from pathlib import Path

catalog_database = 'crawler'
catalog_table = 'responses_raw'

yaml_candidates = [
    Path('../airflow/dags/mongo_catalog_sync_jobs.yaml'),
    Path('/workspace/airflow/dags/mongo_catalog_sync_jobs.yaml'),
    Path('../airflow/dags/mongo_catalog_sync_jobs.yaml'),
    Path('airflow/dags/mongo_catalog_sync_jobs.yaml'),
]
for candidate in yaml_candidates:
    if not candidate.exists():
        continue
    try:
        import yaml
        config = yaml.safe_load(candidate.read_text())
        for job in config.get('jobs', []):
            if job.get('catalog_database') == catalog_database:
                catalog_table = job['catalog_table'].replace('.', '_')
                break
    except Exception:
        pass
    break

full_table_name = f'spark_catalog.{catalog_database}.{catalog_table}'
print(f'Looking for table: {full_table_name}')
print('Tables in warehouse:')
spark.sql(f'SHOW TABLES IN {catalog_database}').show(truncate=False)

if not spark.catalog.tableExists(full_table_name):
    raise ValueError(f'Table {full_table_name} does not exist yet. Run the Mongo sync job first.')


Looking for table: spark_catalog.crawler.responses_raw
Tables in warehouse:
+---------+-------------+-----------+
|namespace|tableName    |isTemporary|
+---------+-------------+-----------+
|crawler  |responses_raw|false      |
+---------+-------------+-----------+



Note that the catalog name is spark_catalog compared to Trino:

`lakehouse.crawler.responses_raw`

In [7]:
print(full_table_name)

spark_catalog.crawler.responses_raw


In [10]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# jobs:
#   - name: crawler_articles
#     mongo_uri: mongodb://mongodb:27017
#     mongo_database: Crawler
#     mongo_collection: crawler_responses_datalake
#     catalog_database: crawler
#     catalog_table: responses_raw
#     schedule: "@hourly"

df = spark.table("spark_catalog.crawler.responses_raw")
print(f'Row count: {df.count()}')

df.show(n=2)




Row count: 2
+-------------------+-------+--------------------+--------------------+--------+--------------------+--------+-------+--------------------+--------------------+
|            Created|Updated|                 Uri|                 _id|entities|                 src|src_type|summary|                text|          _synced_at|
+-------------------+-------+--------------------+--------------------+--------+--------------------+--------+-------+--------------------+--------------------+
|2026.09.20:14:39:43|   NULL|https://www.thegu...|bfe607a9-e00b-431...|    NULL|https://www.thegu...|   media|   NULL|us vice-president...|2026-09-20 19:27:...|
|2026.09.20:14:39:44|   NULL|https://www.thegu...|abebf43f-65da-49b...|    NULL|https://www.thegu...|   media|   NULL|for two decades, ...|2026-09-20 19:27:...|
+-------------------+-------+--------------------+--------------------+--------+--------------------+--------+-------+--------------------+--------------------+



In [11]:
spark.stop()